# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

---

**My lane question is a ranking question** — "which pages should a reviewer look at
first?" — with an observed yes/no label (`is_declining_label`). Following the toolkit's
guidance for that shape, I use classifier *probabilities* as the ranking score and
evaluate with **Precision@K**, not accuracy: the deliverable is an ordered queue, and
P@K is the metric that describes an ordered queue.

**Methods, in the order I tried them:**

1. **Dummy floor** — sklearn's `DummyClassifier` (prior probabilities). Not a model;
   the floor below the floor, so every real number has something honest under it.
2. **Logistic regression** (scaled, class-weighted) — the readable start. Its
   coefficients say *which direction* each signal pushes, in plain terms.
3. **Decision tree** (depth 5) — the printable middle: a reviewer can literally read
   the whole model.
4. **Random forest** (200 trees, depth 10, class-weighted) — the stronger learner,
   kept only if the comparison table earns it.

These are the same three learners as the reference pipeline, on purpose: ML-08 asks me
to compare against my Week-4 baseline on the *same data, same split, same metric*, and
using the pipeline's own learners keeps that comparison clean.

**My one experiment on top:** step 01 of the pipeline already engineers three
availability flags — `has_clicks`, `has_ai_sessions`, `measurable_opportunity` — but
the reference feature list leaves them out. I train a second random forest with those
three flags added and report it as a separate row, caveats attached (Section 4), rather
than quietly swapping it in as "the model".

Seeds are fixed (`random_state=42` everywhere, same as the pipeline) and noted here so
anyone re-running this notebook gets the same table.


In [1]:
# --- Setup: find the repo root, load the prepared feature vector -------------------
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

_root = Path.cwd()
while not (_root / "data" / "raw" / "content_refresh_anonymized.csv").exists() and _root != _root.parent:
    _root = _root.parent
ROOT = _root
sys.path.insert(0, str(ROOT / "scripts"))

from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k

SEED = 42  # fixed everywhere in this notebook, same as the reference pipeline

FEATURE_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
if not FEATURE_PATH.exists():
    import subprocess
    subprocess.run([sys.executable, str(ROOT / "scripts" / "01_prepare_features.py")], check=True)

fv = pd.read_csv(FEATURE_PATH)
y = fv["is_declining_label"].astype(int)
print(f"Feature vector: {len(fv):,} rows x {len(fv.columns)} cols")
print(f"Declining-label base rate (full data): {y.mean():.3f}")

def client_holdout(frame, target, seed=SEED, test_share=0.2):
    """Same split as scripts/03_train_model.py: hold out ~20% of CLIENTS (seed 42),
    so no client's pages appear in both train and test."""
    clients = frame["client_id"].fillna("unknown").astype(str).drop_duplicates().to_numpy()
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(clients)
    n_test = max(1, int(round(len(shuffled) * test_share)))
    test_clients = set(shuffled[:n_test])
    test_mask = frame["client_id"].astype(str).isin(test_clients).to_numpy()
    return np.arange(len(frame))[~test_mask], np.arange(len(frame))[test_mask], test_clients

def build_feature_matrix(frame, extra_flags=()):
    """Same encoding as scripts/03_train_model.py: pipeline numeric features (filled 0)
    + one-hot categoricals. Optionally appends engineered flag columns."""
    numeric = [c for c in MODEL_NUMERIC_FEATURES if c in frame.columns]
    cats = [c for c in MODEL_CATEGORICAL_FEATURES if c in frame.columns]
    num = frame[numeric].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    cat = pd.get_dummies(frame[cats].fillna("unknown").astype(str),
                         prefix=cats, dummy_na=False, dtype=float)
    X = pd.concat([num.reset_index(drop=True), cat.reset_index(drop=True)], axis=1)
    for flag in extra_flags:
        X[flag] = frame[flag].to_numpy()
    return X

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

FLAG_FEATURES = ["has_clicks", "has_ai_sessions", "measurable_opportunity"]
X = build_feature_matrix(fv)
X_flags = build_feature_matrix(fv, extra_flags=FLAG_FEATURES)
print(f"Reference feature matrix: {X.shape[1]} columns")
print(f"With the three availability flags: {X_flags.shape[1]} columns")
print("Excluded on purpose (label-derived): trend_direction, trend_pct")
print("Excluded on purpose (IDs, grouping only): content_id, client_id")

Feature vector: 30,000 rows x 52 cols
Declining-label base rate (full data): 0.542


Reference feature matrix: 52 columns
With the three availability flags: 55 columns
Excluded on purpose (label-derived): trend_direction, trend_pct
Excluded on purpose (IDs, grouping only): content_id, client_id


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

---

**My split is grouped by client** — the same client-holdout the reference pipeline
uses: shuffle the 32 clients with seed 42, hold out ~20% of them (6 clients, 2,325
pages), train on the other 26 clients (27,675 pages). No client's pages appear on
both sides; the code below asserts the overlap is empty.

Why this split is the honest one for my question: pages from one client share
site-level character — templates, topics, measurement setup — so a random row split
would let the model memorise *which client* a page belongs to and inflate the score.
The deployment question is "does the ranking work on pages from a portfolio I haven't
modelled?", and only a grouped split asks that. (In ML-09 I measure exactly how much
the random split inflates the number: P@50 drops from 0.90 to 0.68.)

Why not a time split: the starter CSV is a single snapshot with trailing-90-day
aggregates — there is no per-row date to cut on, so a past→future split cannot be
built from this file honestly. Grouped-by-client is the strongest split the data
supports, and I say so rather than faking a time axis.

One more number to keep in view: the holdout's base rate is **0.391**, lower than the
full-data 0.542, because the six held-out clients happen to decline less. Every P@K
below sits next to that 0.391 — on this slice, random ranking scores ~0.39.


In [2]:
train_idx, test_idx, test_clients = client_holdout(fv, y)
train_clients = set(fv.iloc[train_idx]["client_id"]) 
assert not (train_clients & test_clients), "client leaked across the split!"
print(f"Train: {len(train_idx):,} pages from {len(train_clients)} clients "
      f"(base rate {y.iloc[train_idx].mean():.3f})")
print(f"Test:  {len(test_idx):,} pages from {len(test_clients)} held-out clients "
      f"(base rate {y.iloc[test_idx].mean():.3f})")
print("Client overlap between train and test: 0 (asserted)")

Train: 27,675 pages from 26 clients (base rate 0.555)
Test:  2,325 pages from 6 held-out clients (base rate 0.391)
Client overlap between train and test: 0 (asserted)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

---

The table below is computed in this notebook, in one run, on the holdout from
Section 2 — my ML-07 rule and the reference rule included, so nothing is compared
across different slices. (It reproduces the pipeline's saved numbers exactly:
reference rule P@50 = 0.24, random forest P@50 = 0.68, AUC = 0.747.)

**Reading the table:**

- The **random forest is the model I keep**: P@20 = 0.70, P@50 = 0.68, P@100 = 0.70,
  AUC 0.747 — about **2.8× the reference rule's P@50** and 1.4× my own points rule
  (0.48), on clients it never saw. The depth-5 decision tree is close behind
  (P@50 = 0.66) and is the version I would show a reviewer who wants to read a model.
- Logistic regression (P@50 = 0.40) beats the reference rule but not my points rule —
  a useful reminder that "learned" does not automatically beat "hand-written".
- The dummy floor sits at the base rate (≈0.39 across K; its scores are all
  identical, so its exact top-K is tie-ordering noise), as it must.
- **The flags variant is the biggest number in the table (P@50 = 0.84) and I do not
  fully trust it yet.** Its top 50 turns out to be 49 pages from a single held-out
  client (Section 4 / ML-09), so the gain may be one portfolio's character rather than
  a general improvement. I report it as a directional result that needs a wider
  holdout before it earns a claim — not as my headline.

All figures are saved to `work/outputs/w05_model_metrics.json` as the receipt for
this table.


In [3]:
def my_action_score(df):
    """My hand rule (ML-07): integer points, no fitted weights, no label-derived inputs.
    Returns (score, reason_codes) aligned to df's index."""
    score = pd.Series(0.0, index=df.index)
    codes = pd.Series([[] for _ in range(len(df))], index=df.index, dtype=object)

    def apply(mask, points, code):
        score.loc[mask] += points
        for idx in df.index[mask]:
            codes.at[idx].append(code)

    d = df["days_since_last_update"]; imp = df["impressions_90d"]; pos = df["avg_position"]
    apply(d >= 365, 3, "stale_365_plus")
    apply((d >= 180) & (d < 365), 2, "stale_180_plus")
    apply((d >= 90) & (d < 180), 1, "stale_90_plus")
    apply(imp >= 5000, 3, "visible_high")
    apply((imp >= 500) & (imp < 5000), 2, "visible_mid")
    apply((imp >= 100) & (imp < 500), 1, "visible_low")
    apply((pos > 0) & (pos <= 10) & (df["content_age_days"] >= 180), 2, "page_one_decay_risk")
    apply((pos > 10) & (pos <= 20), 1, "striking_distance")
    apply((df["ctr"] < 0.5) & (imp >= 500), 2, "low_ctr_visible")
    apply((df["word_count"] > 0) & (df["word_count"] < 1200) & (imp >= 250), 1, "thin_visible")
    return score, codes


def suggested_action(codes_str):
    codes = set(codes_str.split("|"))
    if "thin_visible" in codes:
        return "expand_and_refresh"
    if "low_ctr_visible" in codes:
        return "refresh_and_review_ctr"
    if codes & {"stale_365_plus", "stale_180_plus", "stale_90_plus"}:
        return "refresh"
    return "monitor"

def metrics_row(name, y_true, scores):
    return {"model": name,
            "P@20": round(precision_at_k(y_true, scores, 20), 3),
            "P@50": round(precision_at_k(y_true, scores, 50), 3),
            "P@100": round(precision_at_k(y_true, scores, 100), 3),
            "ROC AUC": round(roc_auc_score(y_true, scores), 3),
            "Avg precision": round(average_precision_score(y_true, scores), 3)}

y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
rows = []

# Hand rules on the same holdout
ref = pd.read_csv(ROOT / "data" / "processed" / "baseline_refresh_queue.csv")
ref_scores = fv.iloc[test_idx]["content_id"].map(
    ref.set_index("content_id")["baseline_refresh_score"]).fillna(0).to_numpy()
rows.append(metrics_row("reference_rule (scripts/02)", y_te, ref_scores))
my_scores, _ = my_action_score(fv)
rows.append(metrics_row("my_rule (ML-07 points)", y_te, my_scores.iloc[test_idx].to_numpy()))

# Dummy floor
dummy = DummyClassifier(strategy="prior", random_state=SEED).fit(X.iloc[train_idx], y_tr)
rows.append(metrics_row("dummy (prior)", y_te, dummy.predict_proba(X.iloc[test_idx])[:, 1]))

# Learners
models = {
    "logistic_regression": Pipeline([("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED))]),
    "decision_tree": DecisionTreeClassifier(class_weight="balanced", max_depth=5,
                                            min_samples_leaf=50, random_state=SEED),
    "random_forest": RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                                            min_samples_leaf=25, n_estimators=200,
                                            n_jobs=-1, random_state=SEED),
}
fitted = {}
for name, model in models.items():
    model.fit(X.iloc[train_idx], y_tr)
    fitted[name] = model
    rows.append(metrics_row(name, y_te, model.predict_proba(X.iloc[test_idx])[:, 1]))

# My experiment: random forest + the three availability flags from step 01
rf_flags = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                                  min_samples_leaf=25, n_estimators=200,
                                  n_jobs=-1, random_state=SEED)
rf_flags.fit(X_flags.iloc[train_idx], y_tr)
rows.append(metrics_row("random_forest + has_ flags (experiment)",
                        y_te, rf_flags.predict_proba(X_flags.iloc[test_idx])[:, 1]))

table = pd.DataFrame(rows)
print(f"Holdout base rate: {y_te.mean():.3f}  (random ranking scores ~this at every K)")
print(table.to_string(index=False))

receipt = {"assignment": "ML-08", "seed": SEED, "split": "client_holdout",
           "train_rows": int(len(train_idx)), "test_rows": int(len(test_idx)),
           "test_base_rate": float(y_te.mean()),
           "results": table.to_dict(orient="records")}
out_path = ROOT / "work" / "outputs" / "w05_model_metrics.json"
out_path.write_text(json.dumps(receipt, indent=2))
print(f"\nSaved receipt: {out_path.relative_to(ROOT)}")

Holdout base rate: 0.391  (random ranking scores ~this at every K)
                                  model  P@20  P@50  P@100  ROC AUC  Avg precision
            reference_rule (scripts/02)  0.15  0.24   0.36    0.627          0.468
                 my_rule (ML-07 points)  0.50  0.48   0.44    0.573          0.440
                          dummy (prior)  0.30  0.40   0.37    0.500          0.391
                    logistic_regression  0.35  0.40   0.44    0.700          0.522
                          decision_tree  0.65  0.66   0.68    0.742          0.575
                          random_forest  0.70  0.68   0.70    0.747          0.610
random_forest + has_ flags (experiment)  0.80  0.84   0.79    0.752          0.630

Saved receipt: work/outputs/w05_model_metrics.json


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

---

**What the model leans on.** The random forest's top features are `log_impressions_90d`
(0.132), `days_with_impressions` (0.130), `avg_position` (0.111) and `content_age_days`
(0.091) — visibility and age, the same family of signals my hand rule used, which is
reassuring: the model found the reviewer's intuition, weighted better. No feature is
suspiciously perfect (the label's parents, `trend_direction`/`trend_pct`, are excluded
by construction), and no ID is in the matrix.

**Where it is wrong — false positives (ranked high, not declining).** The top mistakes
are visible keyword articles with real length (2,200–4,500 words) and middling
positions (14–26) that *held up*. They look exactly like declining pages on every
observable I have; what saved them is not in this data. I read these as the honest
ceiling of the feature set, not as bugs.

**Where it is wrong — false negatives (declining, ranked at the bottom).** The missed
pages are nearly invisible: 1–3 impressions in 90 days, `avg_position` 0 (= no data),
CTR 0. A page nobody sees cannot show a measurable decline pattern, so the model
ranks it last. That is a measurement floor: the queue is a *review-priority* tool for
pages with an audience, and it should never be read as "these quiet pages are fine".

**Slices (holdout, from the code below).** Keyword articles: P@50 = 0.68 against a
0.524 slice base rate. Feedly articles are harder: P@50 = 0.52 against a much lower
0.201 base rate — still 2.6× its own base rate, but a reviewer should know the two
content types do not behave alike. By impression tier the model is consistent
(P@50 0.70–0.74 across low/moderate/good tiers).

**The flags experiment, honestly.** Adding `has_clicks`, `has_ai_sessions` and
`measurable_opportunity` moved P@50 from 0.68 to 0.84 — and 49 of its top 50 are one
held-out client. The flags are observable and not label-derived (ML-09 audits them),
so this is not leakage; it is concentration. On a six-client holdout, one client's
character can manufacture a gain like this. My claim is therefore narrow: the flags
are *directionally promising* and worth testing on a wider holdout (or the warehouse
release) before anyone quotes 0.84.


In [4]:
# --- Error analysis on the holdout -----------------------------------------------------
pred = fv.iloc[test_idx].copy()
pred["prob_rf"] = fitted["random_forest"].predict_proba(X.iloc[test_idx])[:, 1]
pred["prob_flags"] = rf_flags.predict_proba(X_flags.iloc[test_idx])[:, 1]

imp = pd.Series(fitted["random_forest"].feature_importances_, index=X.columns).sort_values(ascending=False)
print("Random forest — top 8 features:"); print(imp.head(8).round(3).to_string())

show = ["content_id", "prob_rf", "impressions_90d", "avg_position", "ctr",
        "content_age_days", "word_count", "content_type", "impression_tier"]
fp = pred[pred["is_declining_label"] == 0].sort_values("prob_rf", ascending=False).head(3)
fn = pred[pred["is_declining_label"] == 1].sort_values("prob_rf").head(3)
print("\nTop false positives (ranked high, observed NOT declining):")
print(fp[show].round(3).to_string(index=False))
print("\nFalse negatives (observed declining, ranked lowest):")
print(fn[show].round(3).to_string(index=False))

print("\nP@50 by slice (random forest, holdout):")
for col in ["content_type", "impression_tier"]:
    for key, grp in pred.groupby(col):
        if len(grp) >= 50:
            print(f"  {col} = {key}: n={len(grp)}, base={grp['is_declining_label'].mean():.3f}, "
                  f"P@50={precision_at_k(grp['is_declining_label'], grp['prob_rf'], 50):.3f}")

top50_flags = pred.sort_values("prob_flags", ascending=False).head(50)
print("\nFlags-variant top-50 client mix:", top50_flags["client_id"].value_counts().to_dict())

Random forest — top 8 features:
log_impressions_90d      0.132
days_with_impressions    0.130
avg_position             0.111
content_age_days         0.091
age_tier_365+            0.038
word_count               0.037
log_clicks_90d           0.036
char_count               0.036

Top false positives (ranked high, observed NOT declining):
          content_id  prob_rf  impressions_90d  avg_position  ctr  content_age_days  word_count    content_type impression_tier
content_d2dffcc697a4    0.737             5091          14.1 0.20               144      4496.0 keyword article            good
content_00603b0349b4    0.735             1076          25.6 0.09               125      2439.0 keyword article        moderate
content_e55b8ab078b0    0.734              369          21.8 0.00               112      2192.0 keyword article        moderate

False negatives (observed declining, ranked lowest):
          content_id  prob_rf  impressions_90d  avg_position  ctr  content_age_days  word_coun

  impression_tier = good: n=224, base=0.460, P@50=0.700
  impression_tier = low: n=1608, base=0.333, P@50=0.740
  impression_tier = moderate: n=481, base=0.559, P@50=0.720

Flags-variant top-50 client mix: {'client_f74efabef1': 49, 'client_4fc82b26ae': 1}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

*Final box left unticked on purpose: the notebook is complete and executed in `work/notebooks/`; committing it to my repo is my own submission step.*